# S5 Methods Across Ready Seeds

This notebook builds per-eta eval plots for the S5 method sweeps across local synced run artifacts.

The first code cell is deliberately config-only: no pandas, no Matplotlib, no W&B, and no filesystem walk.

By default, the notebook scans only the imported/cache roots, expands those into exact `out-s5-*` run directories, and then loads just those exact runs.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Keep the first cell purely config-only. We intentionally avoid any filesystem
# walk or directory creation here so this cell stays effectively instant.
MPL_CACHE_DIR = ROOT / "analysis" / "cache" / "matplotlib"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

S5_M = 21
TEACHER_SEED = 20260417
SEEDS = [20260417, 20260418, 20260419]
ETAS = [0.0, 0.1, 0.5, 0.7, 0.8]
SUBSET_SIZE = None
NAIL_REVERSE_MIN_ARTIFACT_UTC = "2026-04-26T00:00:00+00:00"
STRICT_NAIL_REVERSE_COVERAGE = False
REFRESH_WANDB_NAIL_REVERSE = True
ALLOW_WANDB_API = True
SAVE_PLOTS = True
PLOT_EXPORT_DIR = ROOT / "analysis" / "figures" / f"s5_method_seed_sweeps_m{S5_M}_teacher{TEACHER_SEED}"

WANDB_NAIL_REVERSE_RUN_PATHS_BY_ETA = {
    0.0: {
        20260417: "vedsriraman-columbia-university/small-cot-experiments/30da6adf8d23ec01",
        20260418: "vedsriraman-columbia-university/small-cot-experiments/c8e37a94c84ee7c8",
        20260419: "vedsriraman-columbia-university/small-cot-experiments/45d7153ecb0eb2a6",
    },
    0.1: {
        20260417: "vedsriraman-columbia-university/small-cot-experiments/6713631894d0899c",
        20260418: "vedsriraman-columbia-university/small-cot-experiments/9f0dd78c074ab081",
        20260419: "vedsriraman-columbia-university/small-cot-experiments/45c6a318730ea2b9",
    },
    0.5: {
        20260417: "vedsriraman-columbia-university/small-cot-experiments/c9c71ac486830581",
        20260418: "vedsriraman-columbia-university/small-cot-experiments/1cca934dbc1949d0",
        20260419: "vedsriraman-columbia-university/small-cot-experiments/0aa6f3c5e1cb10de",
    },
    0.7: {
        20260417: "vedsriraman-columbia-university/small-cot-experiments/997dc45607ceba97",
        20260418: "vedsriraman-columbia-university/small-cot-experiments/bc41156f510e7889",
        20260419: "vedsriraman-columbia-university/small-cot-experiments/956bf34003ac61ec",
    },
}

# Fast path: search only imported/cache roots, then expand them into exact out-s5-* run dirs.
# This only adds exact local seed-family roots, not a broad reruns crawl.
USE_LOCAL_RERUNS = True
MAX_DISCOVERY_DEPTH = 6

BASE_SEARCH_DIRS = [
    ROOT / "analysis" / "cache" / "s5_online_seed_sweeps" / "dev_node",
    ROOT / "analysis" / "cache" / "s5_online_seed_sweeps" / "aics",
    ROOT / "analysis" / "imports" / "s5_online_seed_sweeps" / "dev_node",
    ROOT / "analysis" / "imports" / "s5_online_seed_sweeps" / "aics",
]

if USE_LOCAL_RERUNS:
    BASE_SEARCH_DIRS.extend([
        ROOT / "reruns" / "s5_m21_teacher20260417",
        ROOT / "reruns" / "s5_m21_teacher20260417_render1337_train20260417",
        ROOT / "reruns" / "s5_m21_teacher20260417_render1337_train20260418",
        ROOT / "reruns" / "s5_m21_teacher20260417_render1337_train20260419",
        ROOT / "reruns" / "s5_m21_teacher20260417_render20260417_train20260417",
        ROOT / "reruns" / "s5_m21_teacher20260417_render20260418_train20260418",
        ROOT / "reruns" / "s5_m21_teacher20260417_render20260419_train20260419",
    ])

# Add exact out-s5-* dirs here if you want to force-include a few runs manually.
MANUAL_SEARCH_DIRS = []

BASE_SEARCH_DIRS = list(dict.fromkeys(path for path in BASE_SEARCH_DIRS if path.exists()))
MANUAL_SEARCH_DIRS = list(dict.fromkeys(path for path in MANUAL_SEARCH_DIRS if path.exists()))

PLOT_EXPORT_DIR


In [ ]:
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import pandas as pd

from scripts.plot_s5_online_seed_sweeps import (
    build_runs_df,
    coverage_table,
    discover_s5_online_runs,
    discover_wandb_s5_online_runs,
    eta_tag,
    expected_seeds_by_eta_for_wandb_overrides,
    iter_run_dirs,
    keep_only_explicit_wandb_for_configured_nail_reverse_etas,
    missing_method_seed_rows,
    plot_per_eta,
    require_method_seed_coverage,
    selected_plot_runs_df,
)

ETA_TAGS = {eta_tag(eta) for eta in ETAS}

def looks_like_requested_online_run(path: Path) -> bool:
    name = path.name
    if not (
        name.startswith("out-s5-nail-")
        or name.startswith("out-s5-opd-")
        or name.startswith("out-s5-noisy-bc-")
    ):
        return False
    if f"m{S5_M}-" not in name:
        return False
    if not any(f"seed{seed}" in name for seed in SEEDS):
        return False
    if not any(f"eta{tag}" in name for tag in ETA_TAGS):
        return False
    return True

SEARCH_DIRS = []
for root in BASE_SEARCH_DIRS:
    for path in iter_run_dirs(root, max_depth=MAX_DISCOVERY_DEPTH):
        if looks_like_requested_online_run(path):
            SEARCH_DIRS.append(path.resolve())
SEARCH_DIRS.extend(path.resolve() for path in MANUAL_SEARCH_DIRS)
SEARCH_DIRS = sorted(dict.fromkeys(SEARCH_DIRS))

pd.DataFrame({
    "search_dir": [str(path) for path in SEARCH_DIRS],
    "parent_root": [str(next((root for root in BASE_SEARCH_DIRS if root in path.parents or path == root), path.parent)) for path in SEARCH_DIRS],
})


In [ ]:
records = []
if SEARCH_DIRS:
    records.extend(
        discover_s5_online_runs(
            SEARCH_DIRS,
            m=S5_M,
            seeds=SEEDS,
            etas=ETAS,
            subset_size=SUBSET_SIZE,
            teacher_seed=TEACHER_SEED,
            max_depth=0,
        )
    )
records.extend(
    discover_wandb_s5_online_runs(
        WANDB_NAIL_REVERSE_RUN_PATHS_BY_ETA,
        m=S5_M,
        seeds=SEEDS,
        etas=ETAS,
        subset_size=SUBSET_SIZE,
        teacher_seed=TEACHER_SEED,
        refresh=REFRESH_WANDB_NAIL_REVERSE,
        allow_api=ALLOW_WANDB_API,
        skip_missing_cache=False,
    )
)

if not records:
    raise RuntimeError("No matching S5 runs were recognized after narrowing to exact out-s5-* dirs.")

runs_df, run_data = build_runs_df(records)
plot_runs_df = selected_plot_runs_df(
    runs_df,
    nail_reverse_min_artifact_utc=NAIL_REVERSE_MIN_ARTIFACT_UTC,
)
plot_runs_df = keep_only_explicit_wandb_for_configured_nail_reverse_etas(
    plot_runs_df,
    WANDB_NAIL_REVERSE_RUN_PATHS_BY_ETA,
)
nail_reverse_required_seeds_by_eta = expected_seeds_by_eta_for_wandb_overrides(
    WANDB_NAIL_REVERSE_RUN_PATHS_BY_ETA,
    default_seeds=SEEDS,
    etas=ETAS,
)
missing_nail_reverse_df = missing_method_seed_rows(
    plot_runs_df,
    method="NAIL-reverse, greedy rollout",
    seeds=nail_reverse_required_seeds_by_eta,
    etas=ETAS,
)
if STRICT_NAIL_REVERSE_COVERAGE:
    require_method_seed_coverage(
        plot_runs_df,
        method="NAIL-reverse, greedy rollout",
        seeds=nail_reverse_required_seeds_by_eta,
        etas=ETAS,
    )
plot_runs_df


In [ ]:
# Missing selected NAIL-reverse runs. Add exact synced run dirs to SEARCH_DIRS if needed.
missing_nail_reverse_df


In [ ]:
# Audit which NAIL-reverse runs were discovered and why they were selected.
included_run_ids = set(plot_runs_df["run_id"])
nail_reverse_audit = runs_df.loc[
    runs_df["method"].eq("NAIL-reverse, greedy rollout")
].copy()
nail_reverse_audit["included_in_plot"] = nail_reverse_audit["run_id"].isin(included_run_ids)
nail_reverse_audit = nail_reverse_audit.sort_values(
    ["eta", "seed", "selection_rank", "artifact_mtime", "final_iter"],
    ascending=[True, True, True, False, False],
)

nail_reverse_audit[
    [
        "eta",
        "seed",
        "subset_size",
        "source_kind",
        "artifact_datetime_utc",
        "reverse_variant",
        "selection_rank",
        "selection_reason",
        "history_restart_count",
        "included_in_plot",
        "completed",
        "final_iter",
        "run_id",
    ]
]

# plot_runs_df is already deduped to the single selected run per eta/seed/method.


In [ ]:
# Coverage summary so we can see which seed/method/eta combinations are already available.
coverage_df = coverage_table(
    plot_runs_df,
    run_data,
    seeds=SEEDS,
    etas=ETAS,
)
coverage_df


In [ ]:
# Save one per-eta reward figure and also show it in the notebook.
plot_per_eta(
    run_data,
    plot_runs_df,
    metric="val/clean_full_exact",
    out_dir=PLOT_EXPORT_DIR if SAVE_PLOTS else None,
    show=True,
)


In [ ]:
# Save one per-eta figure for clean_final_exact and also show it in the notebook.
plot_per_eta(
    run_data,
    plot_runs_df,
    metric="val/clean_final_exact",
    out_dir=PLOT_EXPORT_DIR if SAVE_PLOTS else None,
    show=True,
)
